# 57 · Provider outage and fallback
Level L6 · Part 7 · deep · ~35 min · builds on 37, 15 (also uses 30, 56)

## Goal
Put lesson 37's adapters behind a **failover model** with a **circuit breaker**, run a 10-minute
outage drill on the primary backend, and print errors and latency before, during and after the outage
for three designs: no fallback, fall back on every call, and fall back with a breaker. Then run the
real Scout through the same outage: Gemini goes down, a local model takes over, and Gemini comes back.
The drill measures the latency; the real Scout run checks that the answers stay correct.

## The idea
Lesson 15 retried one provider. That helps when it is busy for a few seconds, but not during an outage
that lasts minutes. Each retry then waits for a timeout before failing. **Provider fallback** sends the
call to a second backend instead. Lesson 37's adapters make that a small change.

Trying the dead provider first on every call is still slow. A **circuit breaker** counts failures.
After a few failures in a row it stops calling the primary for a **cooldown**. Then it lets one trial
call through (**half-open**). If the trial works, traffic goes back to the primary.

Analogy: a house with a generator. When the mains power fails, the switch moves to the generator.
It does not test the mains every second. It checks now and then, and switches back once the power is steady.

## Picture
```mermaid
stateDiagram-v2
    [*] --> closed
    closed --> open: 3 failures in a row
    open --> half_open: cooldown over
    half_open --> closed: trial call works
    half_open --> open: trial call fails
    note right of open: every call goes to the fallback
```

## Step by step
### 1. Start the recorder
Step 10 makes real calls to Gemini and to a local model (Ollama). `llm_replay` recorded them once and
replays them now, so you need neither a key nor Ollama. The outage itself is simulated in Python.
No real provider was harmed.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("57_multi-provider-fallback")

LLM mode: replay (57_multi-provider-fallback.yaml)


### 2. Setup from lesson 37 (shortened)
Scout's notes, folder and tools. `search_notes` matches every word, the calculator uses an allow-list,
and `write_file` stays inside Scout's folder.

In [2]:
import ast, json, operator, os
from pathlib import Path

NOTES = [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
         {"id": 2, "text": "Venus spins backwards compared with most planets."},
         {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
         {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
         {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
         {"id": 6, "text": "Saturn's rings are mostly made of ice."}]
SCOUT_DIR = Path("../data/scout_files"); SCOUT_DIR.mkdir(parents=True, exist_ok=True)
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def search_notes(query: str) -> list[dict]:
    return [n for n in NOTES if all(w in n["text"].lower() for w in query.lower().split())]

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("use numbers, + - * / and brackets, e.g. '24*60+39'")

def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

`write_file`, the schemas, lesson 35's `run_tool` and the unchanged agent loop.

In [3]:
def write_file(filename: str, text: str, overwrite: bool = False) -> str:
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder; use a plain file name")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' exists; pick a new name or set overwrite to true")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

def schema(name, description, **props):
    return {"name": name, "description": description, "parameters": {"type": "object",
            "properties": {k: {"type": t} for k, t in props.items()}, "required": [k for k in props if k != "overwrite"]}}

SCHEMAS = [schema("search_notes", "Search the user's notes for a word or phrase. Returns matching notes.", query="string"),
           schema("calculator", "Evaluate arithmetic like '24*60+39'. Use it for every calculation.", expression="string"),
           schema("write_file", "Save text to a file in Scout's folder.", filename="string", text="string", overwrite="boolean")]
TOOLS = {"search_notes": search_notes, "calculator": calculator, "write_file": write_file}

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

The loop, copied from lesson 37 without changes. Fallback happens behind `model(...)`, so the loop never knows.

In [4]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_day.txt."
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")

### 3. Setup from lesson 37: the two real adapters
The Gemini adapter, as in lesson 37. It keeps Gemini's own turn under `raw`, because Gemini needs it
back unchanged.

In [5]:
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")

def to_gemini(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                  # Gemini's own turn, thought signature included
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The adapter itself sends the request and turns Gemini's reply into Scout's neutral reply.

In [6]:
class GeminiAdapter:
    name = "gemini"
    def __init__(self, model: str = MODEL, system: str = SYSTEM):
        self.model, self.system, self.usage = model, system, []

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=s["name"], description=s["description"],
                                           parameters_json_schema=s["parameters"]) for s in tools]
        cfg = types.GenerateContentConfig(system_instruction=self.system, max_output_tokens=400,
            tools=[types.Tool(function_declarations=decls)],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=self.model, contents=to_gemini(messages), config=cfg)
        u = resp.usage_metadata
        self.usage.append({"input": u.prompt_token_count, "output": u.candidates_token_count or 0})
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        reply = {"tool_calls": calls} if calls else {"text": resp.text or ""}
        return {"role": "assistant", **reply, "raw": resp.candidates[0].content}

The Ollama adapter for the local `granite4.1:3b`, as in lesson 37. Ollama returns tool calls without
ids, so the adapter makes its own (**minted ids**).

In [7]:
import ollama
local = ollama.Client(host="http://localhost:11434")      # creating a client sends nothing

def to_ollama(messages: list[dict], system: str) -> list[dict]:
    out = [{"role": "system", "content": system}]
    for m in messages:
        if m["role"] == "user":
            out.append({"role": "user", "content": m["text"]})
        elif m["role"] == "assistant":
            out.append({"role": "assistant", "content": m.get("text", ""), "tool_calls": [
                {"function": {"name": c["name"], "arguments": c["args"]}} for c in m.get("tool_calls", [])]})
        else:
            out.append({"role": "tool", "tool_name": m["name"], "content": m["content"]})
    return out

The reply half reads tool calls and token counts, and mints an id for each call.

In [8]:
class OllamaAdapter:
    name = "local"
    def __init__(self, model: str = "granite4.1:3b", system: str = SYSTEM):
        self.model, self.system, self.usage, self.minted = model, system, [], 0

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        resp = local.chat(model=self.model, messages=to_ollama(messages, self.system),
                          tools=[{"type": "function", "function": s} for s in tools],
                          options={"temperature": 0, "num_predict": 300})
        self.usage.append({"input": resp.prompt_eval_count or 0, "output": resp.eval_count or 0})
        calls = []
        for tc in resp.message.tool_calls or []:
            self.minted += 1
            calls.append({"id": f"local_{self.minted}", "name": tc.function.name, "args": dict(tc.function.arguments)})
        return {"role": "assistant", **({"tool_calls": calls} if calls else {"text": resp.message.content or ""})}

### 4. Which failures should fail over?
Lesson 15's taxonomy decides. Use the fallback when the provider has a problem: a 5xx, a 429, a
timeout or a lost connection. Do not fall back on a 400. That means **our** request is wrong, and the
fallback would reject it too, or hide the bug. Gemini's SDK raises `errors.ServerError` for 5xx and
`errors.ClientError` for 4xx, both with `.code`.

In [9]:
def should_fail_over(exc: Exception) -> bool:
    if isinstance(exc, errors.ServerError):
        return True                                       # 500, 503, 504: the provider's problem
    if isinstance(exc, errors.ClientError):
        return exc.code == 429                            # 429 yes; 400/401/404: fix the request
    return isinstance(exc, (TimeoutError, ConnectionError))

OUTAGE_503 = errors.ServerError(503, {"error": {"code": 503, "status": "UNAVAILABLE",
                                                "message": "The service is currently unavailable."}})
BAD_400 = errors.ClientError(400, {"error": {"code": 400, "status": "INVALID_ARGUMENT", "message": "bad schema"}})
for exc in (OUTAGE_503, errors.ClientError(429, {"error": {"code": 429}}), BAD_400, TimeoutError("10 s")):
    print(f"{type(exc).__name__:12} {getattr(exc, 'code', ''):>4} -> fail over: {should_fail_over(exc)}")

ServerError   503 -> fail over: True
ClientError   429 -> fail over: True
ClientError   400 -> fail over: False
TimeoutError      -> fail over: True


`OUTAGE_503` is built by hand in the shape Vertex returns. It is our simulated outage, not model
output. We raise it before any HTTP request is sent, so the recording never sees it.

### 5. The circuit breaker
It needs a clock. We use lesson 30's `FakeClock`, a clock you move by hand, so the drill is
deterministic. A ten-minute outage then runs in milliseconds. The breaker has three states. It
records each change with its time, so we can read the timeline afterwards.

In [10]:
class FakeClock:
    """A clock you move by hand (lesson 30)."""
    def __init__(self): self.now = 0.0
    def __call__(self) -> float: return self.now

class CircuitBreaker:
    def __init__(self, clock, threshold: float = 3, cooldown: float = 60.0):
        self.clock, self.threshold, self.cooldown = clock, threshold, cooldown
        self.state, self.failures, self.opened_at, self.events = "closed", 0, 0.0, []

    def _move(self, state: str):
        if state != self.state:
            self.state = state
            self.events.append((self.clock(), state))

    def allow(self) -> bool:                       # may this call go to the primary?
        if self.state == "open" and self.clock() - self.opened_at >= self.cooldown:
            self._move("half_open")                # cooldown over: let one trial through
        return self.state != "open"

    def record_success(self):
        self.failures = 0
        self._move("closed")

    def record_failure(self):
        self.failures += 1
        if self.state == "half_open" or self.failures >= self.threshold:
            self.opened_at = self.clock()
            self._move("open")

### 6. The failover model
`FailoverModel` is itself an adapter (lesson 37's contract), so `run_agent` uses it unchanged. It asks
the breaker, tries the primary, and on a fail-over error uses the fallback.

One rule comes from lesson 37's "leaks". A run that moved to the fallback **stays there** until it
ends. Gemini cannot continue a history whose assistant turns came from another backend, because they
have no `raw`. A new run (one message so far) starts unpinned.

In [11]:
class FailoverModel:
    def __init__(self, primary, fallback, breaker: CircuitBreaker):
        self.primary, self.fallback, self.breaker = primary, fallback, breaker
        self.name, self.served, self.pinned = f"{primary.name}>{fallback.name}", [], False

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        if len(messages) == 1:
            self.pinned = False                        # a new run may use the primary again
        if not self.pinned and self.breaker.allow():
            try:
                reply = self.primary(messages, tools)
                self.breaker.record_success()
                self.served.append(self.primary.name)
                return reply
            except Exception as e:
                if not should_fail_over(e):
                    raise                              # our bug: no fallback hides it
                self.breaker.record_failure()
        self.pinned = True
        self.served.append(self.fallback.name)
        return self.fallback(messages, tools)

### 7. A simulated pair of backends
For the drill, two stand-ins for the real backends. Each one "takes" its recorded per-call time on the
fake clock. Gemini's time comes from lesson 55's 20 runs. The local model's time comes from lesson 16's
two granite rows. Both are recorded outputs. When the primary is down, it hangs until the 10-second
per-call timeout (lesson 30), then fails. That is the worst kind of outage: slow, not a quick 503.

In [12]:
import math, statistics

calls_55 = [c for r in json.loads(Path("../data/recorded/55.json").read_text())["eval"] for c in r["calls"]]
rows_16 = json.loads(Path("../data/recorded/16.json").read_text())["rows"]
SECONDS = {"gemini": statistics.mean(c["seconds"] for c in calls_55),                       # recorded output
           "local": statistics.mean(r["seconds"] for r in rows_16 if r["model"] == "granite4.1:3b")}
TIMEOUT = 10.0
clock = FakeClock()

class SimBackend:
    def __init__(self, name: str):
        self.name, self.down, self.attempts = name, False, 0
    def __call__(self, messages, tools) -> dict:
        self.attempts += 1
        clock.now += TIMEOUT if self.down else SECONDS[self.name]
        if self.down:
            raise TimeoutError(f"{self.name}: no reply in {TIMEOUT:.0f} s")
        return {"role": "assistant", "text": f"(answer from {self.name})"}

print({k: round(v, 2) for k, v in SECONDS.items()}, "seconds per call")

{'gemini': 0.59, 'local': 1.71} seconds per call


### 8. The outage drill
One question every 15 seconds for 10 minutes, 40 requests. The primary is down from t=150 s to t=400 s.
Each request is tagged with a phase (before, during, after). We record whether it worked, how long it
took and who answered. We sent one request at a time, so this drill shows latency, not throughput.

In [13]:
OUTAGE = (150, 400)

def drill(make) -> tuple[list[dict], SimBackend]:
    primary, fallback = SimBackend("gemini"), SimBackend("local")
    model, rows = make(primary, fallback), []
    for arrival in range(0, 600, 15):
        clock.now = float(arrival)
        primary.down = OUTAGE[0] <= arrival < OUTAGE[1]
        phase = "before" if arrival < OUTAGE[0] else ("during" if primary.down else "after")
        try:
            model([{"role": "user", "text": QUESTION}], SCHEMAS)
            ok, by = True, model.served[-1] if hasattr(model, "served") else model.name
        except Exception as e:
            ok, by = False, type(e).__name__
        rows.append({"t": arrival, "phase": phase, "ok": ok, "seconds": clock.now - arrival, "by": by})
    return rows, primary

The summary per phase: requests, errors, median (p50) and p95 latency (lesson 56's nearest-rank
percentile), and the share the fallback served.

In [14]:
def percentile(xs: list[float], p: float) -> float:
    xs = sorted(xs)
    return xs[math.ceil(p / 100 * len(xs)) - 1]

def summary(label: str, rows: list[dict]) -> None:
    print(f"{label}\n  {'phase':7} {'reqs':>4} {'errors':>6} {'p50 s':>6} {'p95 s':>6} {'fallback':>8}")
    for phase in ("before", "during", "after"):
        rs = [r for r in rows if r["phase"] == phase]
        secs = [r["seconds"] for r in rs]
        fb = sum(r["by"] == "local" for r in rs) / len(rs)
        print(f"  {phase:7} {len(rs):>4} {sum(not r['ok'] for r in rs):>6} "
              f"{percentile(secs, 50):>6.2f} {percentile(secs, 95):>6.2f} {fb:>8.0%}")

**Design A: no fallback.** Scout calls Gemini only. Every request during the outage waits 10 seconds,
then fails.

In [15]:
DRILLS = {}
DRILLS["A no fallback"] = drill(lambda primary, fallback: primary)
summary("A no fallback", DRILLS["A no fallback"][0])

A no fallback
  phase   reqs errors  p50 s  p95 s fallback
  before    10      0   0.59   0.59       0%
  during    17     17  10.00  10.00       0%
  after     13      0   0.59   0.59       0%


**Design B: fall back on every call.** A breaker whose threshold is infinite never opens. So every
request first waits out the dead primary, then goes to the fallback.

In [16]:
DRILLS["B fallback per call"] = drill(
    lambda p, f: FailoverModel(p, f, CircuitBreaker(clock, threshold=math.inf)))
summary("B fallback per call", DRILLS["B fallback per call"][0])

B fallback per call
  phase   reqs errors  p50 s  p95 s fallback
  before    10      0   0.59   0.59       0%
  during    17      0  11.71  11.71     100%
  after     13      0   0.59   0.59       0%


No errors. But every answer during the outage took 10 s plus the local call. **Design C: with a
breaker** (3 failures in a row, 60 s cooldown).

In [17]:
DRILLS["C breaker"] = drill(lambda p, f: FailoverModel(p, f, CircuitBreaker(clock, threshold=3, cooldown=60)))
summary("C breaker", DRILLS["C breaker"][0])

C breaker
  phase   reqs errors  p50 s  p95 s fallback
  before    10      0   0.59   0.59       0%
  during    17      0   1.71  11.71     100%
  after     13      0   0.59   0.59       0%


The median during the outage dropped to the local model's 1.71 s, but the p95 is still 11.71 s: some
requests paid the timeout. Which ones? The breaker's timeline tells. We rebuild design C and keep a
handle on its breaker.

In [18]:
holder = {}
def make_c(p, f):
    holder["breaker"] = CircuitBreaker(clock, threshold=3, cooldown=60)
    return FailoverModel(p, f, holder["breaker"])

rows_c, _ = drill(make_c)
print("breaker timeline:", [(int(t), s) for t, s in holder["breaker"].events])
print("requests that waited for the timeout:", [r["t"] for r in rows_c if r["seconds"] > TIMEOUT])

breaker timeline: [(190, 'open'), (255, 'half_open'), (265, 'open'), (330, 'half_open'), (340, 'open'), (405, 'half_open'), (405, 'closed')]
requests that waited for the timeout: [150, 165, 180, 255, 330]


Read it with the picture (times are when a call ended). Three timeouts (t=150, 165, 180) open the
breaker at 190. After each 60 s cooldown, one half-open trial (t=255, 330) pays the timeout and
reopens it. The trial at t=405, the first request after Gemini came back, works and closes it.

### 9. Compare the three designs
The outage numbers that matter: failed requests, p95 latency, how many calls were wasted on the dead
primary, and when Gemini served again after it came back at t=400.

In [19]:
print(f"{'design':20} {'errors':>6} {'p95 during':>10} {'wasted calls':>12} {'back on gemini at':>17}")
for label, (rows, primary) in DRILLS.items():
    during = [r for r in rows if r["phase"] == "during"]
    wasted = sum(r["seconds"] >= TIMEOUT for r in during)
    back = next((r["t"] for r in rows if r["t"] >= OUTAGE[1] and r["by"] == "gemini"), None)
    print(f"{label:20} {sum(not r['ok'] for r in during):>6} {percentile([r['seconds'] for r in during], 95):>10.2f}"
          f" {wasted:>12} {('t=' + str(back)) if back is not None else 'never':>17}")

design               errors p95 during wasted calls back on gemini at
A no fallback            17      10.00           17             t=405
B fallback per call       0      11.71           17             t=405
C breaker                 0      11.71            5             t=405


The breaker adds no errors and cuts the calls wasted on the dead primary from 17 to 5, so the median
outage request takes 1.71 s instead of 11.71 s. Its p95 is unchanged: 5 of 17 requests still paid the
timeout. All designs are back on Gemini at t=405. Here the trial was due just then; in the worst case
recovery waits up to one cooldown. Two settings trade off against each other. A shorter cooldown
finds the recovery sooner but wastes more trial calls. A lower threshold trips sooner but may also trip
on a short blip that a retry would have fixed (lesson 14).

### 10. The same outage, real Scout
Now with real models. The primary is Gemini behind an `OutageSwitch`. When the switch is down, it raises
the hand-built 503 before any request leaves. The fallback is the local granite model. We have only four
runs, so the breaker opens after **one** failure, with a 60 s cooldown on a fake clock that we move by hand.

The real runs report errors, backend and correctness, not latency. Replayed calls return instantly, so
their wall-clock times would mean nothing. Latency comes from the drill, which uses the recorded
per-call times.

In [20]:
class OutageSwitch:
    def __init__(self, adapter):
        self.adapter, self.name, self.down = adapter, adapter.name, False
    def __call__(self, messages, tools):
        if self.down:
            raise OUTAGE_503                       # simulated: nothing is sent
        return self.adapter(messages, tools)

real_clock = FakeClock()
gemini = OutageSwitch(GeminiAdapter())
scout_model = FailoverModel(gemini, OllamaAdapter(), CircuitBreaker(real_clock, threshold=1, cooldown=60))

def ask(label: str) -> dict:
    for old in SCOUT_DIR.glob("*.txt"):
        old.unlink()
    start = len(scout_model.served)
    run = run_agent(QUESTION, scout_model, TOOLS, SCHEMAS)
    saved = (SCOUT_DIR / "mars_day.txt").read_text() if (SCOUT_DIR / "mars_day.txt").exists() else ""
    correct = all("1479" in (s or "").replace(",", "") for s in (run["answer"], saved))
    return {"run": label, "served": scout_model.served[start:], "errors": run["errors"], "correct": correct}

Run 1, **before**: Gemini is up.

In [21]:
REAL = [ask("1 before")]
print(REAL[-1])

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: calculator({'expression': '24 * 60 + 39'}) -> 1479
step 3: write_file({'overwrite': True, 'text': 'A day on Mars is 1479 minutes long.', 'filename': 'mars_day.txt'}) -> "saved 35 characters to mars_day.txt"
step 4: answer
{'run': '1 before', 'served': ['gemini', 'gemini', 'gemini', 'gemini'], 'errors': 0, 'correct': True}


Run 2, **during**: the outage starts. The first call fails, the breaker opens, and the local model
answers the whole run.

In [22]:
gemini.down = True
REAL.append(ask("2 during"))
print(REAL[-1], "| breaker:", scout_model.breaker.state)

step 1: search_notes({'query': 'Mars day'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60 + 39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': '1479'}) -> "saved 4 characters to mars_day.txt"
step 4: answer
{'run': '2 during', 'served': ['local', 'local', 'local', 'local'], 'errors': 0, 'correct': True} | breaker: open


Run 3, 30 s later, still **during**. The breaker is open and the cooldown is not over, so Scout goes
straight to the local model. It does not wait on Gemini.

In [23]:
real_clock.now += 30
REAL.append(ask("3 during"))
print(REAL[-1], "| breaker:", scout_model.breaker.state)

step 1: search_notes({'query': 'Mars day'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60 + 39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': '1479'}) -> "saved 4 characters to mars_day.txt"
step 4: answer
{'run': '3 during', 'served': ['local', 'local', 'local', 'local'], 'errors': 0, 'correct': True} | breaker: open


Run 4, two minutes later, **after**: Gemini is back. The cooldown is over, so the first call is the
half-open trial. It works and closes the breaker.

In [24]:
gemini.down, real_clock.now = False, real_clock.now + 120
REAL.append(ask("4 after"))
print(REAL[-1], "| breaker:", scout_model.breaker.state)

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: calculator({'expression': '24*60+39'}) -> 1479


step 3: write_file({'text': 'A day on Mars is 1479 minutes long.', 'filename': 'mars_day.txt', 'overwrite': True}) -> "saved 35 characters to mars_day.txt"
step 4: answer
{'run': '4 after', 'served': ['gemini', 'gemini', 'gemini', 'gemini'], 'errors': 0, 'correct': True} | breaker: closed


The whole real drill in one table (the 503 is simulated; the model answers are recorded). If the model
cells were skipped, the table is empty.

In [25]:
print(f"{'run':9} {'answered by':28} {'tool errors':>11}  correct")
for r in REAL if "REAL" in globals() else []:
    print(f"{r['run']:9} {' '.join(r['served']):28} {r['errors']:>11}  {r['correct']}")
print("breaker timeline:", [(int(t), s) for t, s in scout_model.breaker.events])

run       answered by                  tool errors  correct
1 before  gemini gemini gemini gemini            0  True
2 during  local local local local                0  True
3 during  local local local local                0  True
4 after   gemini gemini gemini gemini            0  True
breaker timeline: [(0, 'open'), (150, 'half_open'), (150, 'closed')]


All four runs were correct with no tool errors. Runs 2 and 3 were answered entirely by the local model,
and run 3 never touched Gemini. Run 4 was back on Gemini from its first call. The timeline shows the
breaker opening at t=0 (run 2), the half-open trial at t=150 (run 4) and the close right after.

The fallback is not identical. Gemini saved a sentence ("A day on Mars is 1479 minutes long.") and set
`overwrite`. The local model saved only "1479". Both pass our check, but a user would notice. That is
why a fallback model needs its own evals (lesson 48).

## At real scale
- **Gateways ship this built in.** LiteLLM has ordered `fallbacks` and puts a failing deployment in
  cooldown (example config: `allowed_fails: 3`, `cooldown_time: 30`). OpenRouter first tries providers
  with no major failure in roughly the last 30 s; `allow_fallbacks` defaults to true.
- **The fallback inherits the whole load**, at the busiest minute. Size its quota for that (lesson 56),
  or it fails with 429s right after the primary.
- **The fallback is a different model.** Run the same evals on it (lesson 48). A fallback serving 30%
  of traffic for a week is no longer a fallback.
- **Shared outages happen.** Anthropic's 529 comes "when the API experiences high traffic across all
  users"; two models from one vendor can fail together. Fail over across vendors or regions.
- **Watch:** breaker changes per backend, fallback share, errors and p95 per phase, time to recover.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| Short blips (single 429/503) | Retry with backoff (lesson 14), no failover | Cheapest; same model, same quality |
| Outage of minutes or longer | Failover with a circuit breaker | No errors, and only a few calls wasted on the dead provider |
| Fallback model much weaker | Fall back to a degraded answer or a queue (lesson 15's busy message) | A wrong answer can be worse than a late one |
| Bad request (400, schema) | Raise, never fail over | It fails everywhere, or the fallback hides the bug |
| Many teams, many providers | A shared gateway (LiteLLM, OpenRouter) with one policy | One place for cooldowns, quotas and logs |
| Long multi-step runs | Pin a run to one backend once it moves | Vendor-only state (`raw`, ids) does not transfer |

## What just happened
- `should_fail_over` sent 5xx, 429 and timeouts to the fallback, and raised on 400s.
- A three-state `CircuitBreaker` on a fake clock decided when the primary may be tried again.
- `FailoverModel` is an adapter, so lesson 37's loop ran unchanged. It pins a run to the fallback once
  the run moves there.
- A 10-minute drill compared three designs: no fallback lost all 17 outage requests, per-call fallback
  lost none but waited 10 s on each, and the breaker lost none and wasted 5 calls.
- The real Scout went from Gemini to the local model and back, with the breaker's timeline to prove it.

## Common mistakes
**1. Switching back to Gemini in the middle of a run.** Once the local model has answered a step, the
history holds an assistant turn without Gemini's `raw`. Sending that history to Gemini fails inside
the adapter. Here is a history as the local model would leave it, built by hand.

In [26]:
mixed = [{"role": "user", "text": QUESTION},
         {"role": "assistant", "tool_calls": [{"id": "local_1", "name": "search_notes", "args": {"query": "Mars day"}}]},
         {"role": "tool", "call_id": "local_1", "name": "search_notes", "content": "[]", "is_error": False}]
to_gemini(mixed)

KeyError: 'raw'

Fix: pin the run, as `FailoverModel` does, and go back to the primary only at the start of a new run.

In [27]:
pin_demo = FailoverModel(SimBackend("gemini"), SimBackend("local"), CircuitBreaker(clock, threshold=1))
pin_demo.primary.down = True
pin_demo(mixed[:1], SCHEMAS)                      # step 1 of a run: primary fails, fallback answers
pin_demo.primary.down = False
pin_demo(mixed, SCHEMAS)                          # step 2, primary is back: still the fallback
print(pin_demo.served, "| primary attempts:", pin_demo.primary.attempts)

['local', 'local'] | primary attempts: 1


**2. A breaker that never closes.** With no half-open trial (an infinite cooldown), the breaker trips
during the outage and then stays open. Scout runs on the slower fallback long after Gemini is back,
and no error ever shows it. A drill check catches it.

In [28]:
rows_stuck, _ = drill(lambda p, f: FailoverModel(p, f, CircuitBreaker(clock, threshold=3, cooldown=math.inf)))
after = [r for r in rows_stuck if r["phase"] == "after"]
assert any(r["by"] == "gemini" for r in after), f"{sum(r['by'] == 'local' for r in after)}/{len(after)} after-outage requests still on the fallback"

AssertionError: 13/13 after-outage requests still on the fallback

Fix: a finite cooldown with a half-open trial, as in design C. Then the same check passes.

In [29]:
after = [r for r in DRILLS["C breaker"][0] if r["phase"] == "after"]
print(f"after the outage: {sum(r['by'] == 'gemini' for r in after)}/{len(after)} requests back on gemini")

after the outage: 13/13 requests back on gemini


## Try it
A provider can also **flap**: every other call fails. Then "3 failures in a row" never happens, so
design C's breaker never opens, and half the calls wait for the timeout. Write `RateBreaker(clock,
window=10, max_error_rate=0.5, min_calls=6, cooldown=60)`. It has the same methods as `CircuitBreaker`
(`allow`, `record_success`, `record_failure`, `state`, `events`). It opens when at least half of the
last 10 calls failed, given at least 6 calls. Feed both breakers the outcomes `[True, False] * 10`
(True = success) and show that only yours opens.

In [30]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [31]:
llm_replay.stop()

## Key terms
- **provider fallback (failover)** — sending a model call to a second backend when the first has a provider-side failure (5xx, 429, timeout), behind the same adapter contract.
- **circuit breaker** — a guard that stops calling a failing backend after a run of failures, waits a cooldown, then lets one trial call decide whether to resume.
- **half-open** — the breaker state after the cooldown, where a single trial call goes to the primary: success closes the breaker, failure reopens it.
- **cooldown** — how long an open breaker keeps a backend out of rotation before the next trial call.
- **pinned run** — a run that stays on the backend it moved to until it ends, because vendor-only state (Gemini's `raw` turn, call ids) does not transfer mid-run.
- **outage drill** — a timed chaos drill that takes a provider down for a window and compares errors and latency before, during and after.